# DATA3404 Assignment 2 - Bootstrap Code

# Please do not run this unnecessarily - the more it is run the more likely rate limiting will apply to the associated files for everyone. Your files are retained after you downloaded them with this, so you don't have to re-run this notebook. 
**Instead, once the Bootsrap notebook has been executed and all files be downloaded, use the Assignment2_Databricks_SQL.sql file to start your assignment work.**

## Utility Functions

#### Use a modified version of GoogleDriveDownloader.
The following defines a GoogleDriveDownloader that allows to download data files from Google docs into the Databricks file system (DBFS). These functions are used later in this notebook.

In [0]:
from __future__ import print_function
import requests
import zipfile
import warnings
from sys import stdout
from os import makedirs
from os.path import dirname
from os.path import exists

class GoogleDriveDownloader:
    CHUNK_SIZE = 32768
    DOWNLOAD_URL = 'https://drive.usercontent.google.com/download'

    @staticmethod
    def download_file_from_google_drive(file_id, dest_path, overwrite=False, unzip=False):
        destination_directory = dirname(dest_path)
        if not exists(destination_directory):
            makedirs(destination_directory)

        if not exists(dest_path) or overwrite:
            session = requests.Session()
            print('Downloading {} into {}... '.format(file_id, dest_path), end='')
            stdout.flush()
            params = {'id': file_id, 'export': "download", 'confirm': "t"}
            response = session.get(GoogleDriveDownloader.DOWNLOAD_URL, params=params, stream=True)
            GoogleDriveDownloader._save_response_content(response, dest_path)
            print('Done.')

            if unzip:
                try:
                    print('Unzipping...', end='')
                    stdout.flush()
                    with zipfile.ZipFile(dest_path, 'r') as z:
                        z.extractall(destination_directory)
                    print('Done.')
                except zipfile.BadZipfile:
                    warnings.warn('Ignoring `unzip` since "{}" does not look like a valid zip file'.format(file_id))

    @staticmethod
    def _save_response_content(response, destination):
        with open(destination, 'wb') as f:
            for chunk in response.iter_content(GoogleDriveDownloader.CHUNK_SIZE):
                if chunk:  # filter out keep-alive new chunks
                    f.write(chunk)

Next define a utility function 'dbfs_file_exists()' that correctly checks whether a file exists in Databricks' DBFS:

In [0]:
# check for existing file in Databricks Volume / DBFS
def dbfs_file_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception as e:
        if 'java.io.FileNotFoundException' in str(e) or 'com.databricks.sql.io.CloudFileNotFoundException' in str(e):
            return False
        else:
            raise

## Retrieve data from Google Drive and upload into a Databricks Unity Catalog Volume.
We perform these steps initially so that the data persists. 

First we define which files we want to retrieve from the lecture's Google Drive folder.

In [0]:
unity_volume_path = "/Volumes/workspace/data3404/airbnb/"
prefix = "airbnb"
files = [
  { "name": f"{prefix}_cities.csv", "file_id": "1Ku-TEkQV2cBkwxrryV_-VpN5kx_CbFku" },
  { "name": f"{prefix}_neighbourhoods.csv", "file_id": "1tBdibXa5CpZ3N43t4yXG5uyHzXrI59q8" },
  { "name": f"{prefix}_hosts.csv", "file_id": "1RYtXThxcoHYUHEZF8taED_w50h5M3P3d" },
  { "name": f"{prefix}_hosts-small.csv", "file_id": "1sUhW7O_CZ_Evs2QEkL1SW5c-xutpNZfq"},
  { "name": f"{prefix}_hosts-medium.csv", "file_id": "1cdM9b7siS8KYmhqKADPnuYYhnkrQzmoZ"},
  { "name": f"{prefix}_listings-small.csv", "file_id": "1NiWcQXw3SUP_bXjbHhSgggQXqcG_fJeW" },
  { "name": f"{prefix}_listings-medium.csv",   "file_id": "1fQO9mdxz_2mmhqTwxifYcIn7GsrSKOXV" },
  { "name": f"{prefix}_listings-large.csv", "file_id": "1kIZ50FQoggDa-DXGnkLH2SQQH5I4gCb8" },
  { "name": f"{prefix}_reviews-small.csv", "file_id": "1jGzRJJ9wQV-_N8AJVGtNfeRAeuYQShOj" },
  { "name": f"{prefix}_reviews-medium.csv",  "file_id": "1OoMaDWQDbsaxb1bNB2Lqd4pVihpdoWSz" },
  { "name": f"{prefix}_reviews-large.csv", "file_id": "1Ix25lM_RWzh2HjugDWA5hxcohBtCqxy-" }
]

**Preparation Step 1:** Create a Unity Catalog Volume for Assignment 2 (if it not exists already).

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.data3404;
CREATE VOLUME IF NOT EXISTS workspace.data3404.airbnb;

**Preparation Step 2:** Remove any existing files (in case this notebook is re-run)

In [0]:
import os
for file in files:
  dbutils.fs.rm(unity_volume_path+file["name"])

### Download datasets to your Unity Catalog Volume
We download tiny, small, medium and large (1GB) datasets to your Databricks volume,
but only for those files that have not been loaded yet (to save bandwidth).

If this includes the large datafile, may take 20 to 30 seconds to complete...

In [0]:
for file in files:
  if not dbfs_file_exists(unity_volume_path+file["name"]):
    GoogleDriveDownloader.download_file_from_google_drive(file_id=file["file_id"], dest_path=unity_volume_path+file["name"])

### Check the files in Databricks' Volume.
You can use the below cell to explore the Unity Volume directories and check that the files are where they should be, and contain the correct data.

**Note:** If any file shows a _size_ of either 0 or only about 2448 bytes, then something went wrong during the download.

In [0]:
display(dbutils.fs.ls(unity_volume_path))
display(dbutils.fs.head(unity_volume_path+"airbnb_listings-small.csv", 290))

Congratulations, you have now all the datasets which you need for the Assignment 2 available.

You can now close this notebook and continue with the **"Assignment2_Databricks_SQL"** notebook about how to access these datasets using SQL.